# Seção 2 (básico) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`02_criando_um_modelo`](02_criando_um_modelo.ipynb), [`09_inspecionando_campos`](09_inspecionando_campos.ipynb) | `instancia.model_fields` depreciado (2.11+): use a classe |
| [`04_serializacao`](04_serializacao.ipynb) | a v2 não usa `json.dumps` por baixo; o exemplo "JSON" do original gerava `dict` |
| [`06_campos_obrigatorios_e_opcionais`](06_campos_obrigatorios_e_opcionais.ipynb) | `user_list or []` troca a lista vazia do chamador por outra |
| [`13_projeto_solucao_comentada`](13_projeto_solucao_comentada.ipynb) | `str = None` rejeita `null` explícito: o certo é `str \| None = None` |

## Novas

**1. `exclude_none` e `exclude_defaults` apagam o que o cliente mandou de propósito.**

In [1]:
from decimal import Decimal
from pydantic import BaseModel, ValidationError


class Preferences(BaseModel):
    theme: str = "light"
    nickname: str | None = None


sent = Preferences(theme="light", nickname=None)          # o cliente ESCOLHEU light e ESCOLHEU limpar o apelido
print(f"sent.model_fields_set: {sorted(sent.model_fields_set)}")
print(f"exclude_none=True: {sent.model_dump(exclude_none=True)}")
print(f"exclude_defaults=True: {sent.model_dump(exclude_defaults=True)}")
print(f"exclude_unset=True: {sent.model_dump(exclude_unset=True)}")

sent.model_fields_set: ['nickname', 'theme']
exclude_none=True: {'theme': 'light'}
exclude_defaults=True: {}
exclude_unset=True: {'theme': 'light', 'nickname': None}


Os dois primeiros filtram pelo **valor**; só o `exclude_unset` filtra pelo que foi **enviado**. Para atualizar um registro
("limpar o apelido"), o `exclude_none` descarta justamente a mudança pedida.

**2. Campo `int` aceita `True`.**

In [2]:
class Counter(BaseModel):
    clicks: int


json_payload = '{"clicks": true}'
print(f"Counter(clicks=True): {Counter(clicks=True)!r}")
print(f"Counter.model_validate_json(json_payload): {Counter.model_validate_json(json_payload)!r}")

Counter(clicks=True): Counter(clicks=1)
Counter.model_validate_json(json_payload): Counter(clicks=1)


`bool` é subclasse de `int` em Python, e o modo *lax* aceita a conversão. Um formulário que manda `true` num campo numérico passa
como `1`. Em campos onde isso importa: `Field(strict=True)` ou `StrictInt`.

**3. O JSON Schema de entrada e o de saída são diferentes.**

In [3]:
class Price(BaseModel):
    amount: Decimal


print(f"modo validation: {Price.model_json_schema(mode='validation')['properties']['amount']}")
print(f"modo serialization: {Price.model_json_schema(mode='serialization')['properties']['amount']}")

modo validation: {'anyOf': [{'type': 'number'}, {'pattern': '^(?!^[-+.]*$)[+-]?0*\\d*\\.?\\d*$', 'type': 'string'}], 'title': 'Amount'}
modo serialization: {'pattern': '^(?!^[-+.]*$)[+-]?0*\\d*\\.?\\d*$', 'title': 'Amount', 'type': 'string'}


Na entrada, `Decimal` aceita número **ou** string; na saída, é sempre string. Documentar a API (OpenAPI) com o schema errado faz
o cliente gerado esperar número na resposta. O FastAPI usa os dois modos; quem publica o schema à mão precisa escolher.

## Veja também

- [`pydantic/01_pydantic_basico`](../../01_pydantic_basico.ipynb) (seção de caveats): `json.dumps(model_dump())` com UUID/datetime,
  chave com erro de digitação ignorada, datetime com e sem fuso, `model_validate(instância)` sem revalidar.